In [2]:
import os
import sys

# Îi spunem clar lui Spark: "Folosește DOAR Python-ul din Anaconda (test_spark)!"
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Read Inside Airbnb data") \
    .getOrCreate()


In [12]:
listings = spark.read.csv("data/listings.csv.gz",
                          header= True,
                          inferSchema=True,
                          sep=",", #SEPARATOR
                          quote='"', #pentru valorile string
                          escape='"',
                          multiLine=True, 
                          mode="PERMISSIVE"
                         )

In [13]:
for field in listings.schema:
    print(field)

StructField('id', LongType(), True)
StructField('listing_url', StringType(), True)
StructField('scrape_id', LongType(), True)
StructField('last_scraped', DateType(), True)
StructField('source', StringType(), True)
StructField('name', StringType(), True)
StructField('description', StringType(), True)
StructField('neighborhood_overview', StringType(), True)
StructField('picture_url', StringType(), True)
StructField('host_id', IntegerType(), True)
StructField('host_url', StringType(), True)
StructField('host_name', StringType(), True)
StructField('host_since', DateType(), True)
StructField('host_location', StringType(), True)
StructField('host_about', StringType(), True)
StructField('host_response_time', StringType(), True)
StructField('host_response_rate', StringType(), True)
StructField('host_acceptance_rate', StringType(), True)
StructField('host_is_superhost', StringType(), True)
StructField('host_thumbnail_url', StringType(), True)
StructField('host_picture_url', StringType(), True)


In [14]:
neighbourhoods = listings.select(listings.neighbourhood_cleansed)
neighbourhoods.show(20)

+----------------------+
|neighbourhood_cleansed|
+----------------------+
|             Islington|
|  Kensington and Ch...|
|           Westminster|
|            Wandsworth|
|         Tower Hamlets|
|  Richmond upon Thames|
|              Haringey|
|  Hammersmith and F...|
|  Hammersmith and F...|
|             Southwark|
|           Westminster|
|                Barnet|
|              Hounslow|
|             Southwark|
|        Waltham Forest|
|                Barnet|
|  Hammersmith and F...|
|  Hammersmith and F...|
|                 Brent|
|                Camden|
+----------------------+
only showing top 20 rows


In [15]:
neighbourhoods = listings.select(listings.neighbourhood_cleansed)
neighbourhoods.show(20, truncate = False)

+----------------------+
|neighbourhood_cleansed|
+----------------------+
|Islington             |
|Kensington and Chelsea|
|Westminster           |
|Wandsworth            |
|Tower Hamlets         |
|Richmond upon Thames  |
|Haringey              |
|Hammersmith and Fulham|
|Hammersmith and Fulham|
|Southwark             |
|Westminster           |
|Barnet                |
|Hounslow              |
|Southwark             |
|Waltham Forest        |
|Barnet                |
|Hammersmith and Fulham|
|Hammersmith and Fulham|
|Brent                 |
|Camden                |
+----------------------+
only showing top 20 rows


In [17]:
review_locations = listings.select(listings.review_scores_location)
review_locations.show()

+----------------------+
|review_scores_location|
+----------------------+
|                  4.78|
|                  4.93|
|                  4.89|
|                   4.6|
|                  4.85|
|                   4.9|
|                  4.77|
|                  4.53|
|                  4.79|
|                  4.79|
|                   4.5|
|                  4.64|
|                  4.84|
|                  4.86|
|                   4.0|
|                  4.75|
|                  NULL|
|                  4.66|
|                  4.67|
|                   5.0|
+----------------------+
only showing top 20 rows


In [21]:
high_score_listings = listings \
    .filter(listings.review_scores_location > 4.5) \
    .select('id', 'price', 'name', 'review_scores_location')

high_score_listings.show(20, truncate=False)

+-----+-------+-------------------------------------------------+----------------------+
|id   |price  |name                                             |review_scores_location|
+-----+-------+-------------------------------------------------+----------------------+
|13913|$70.00 |Holiday London DB Room Let-on going              |4.78                  |
|15400|$149.00|Bright Chelsea  Apartment. Chelsea!              |4.93                  |
|17402|$411.00|Very Central Modern 3-Bed/2 Bath By Oxford St W1 |4.89                  |
|24328|NULL   |Battersea live/work artist house                 |4.6                   |
|36274|$210.00|Bright 1 bedroom apt off brick lane in Shoreditch|4.85                  |
|36299|$280.00|Kew Gardens 3BR house in cul-de-sac              |4.9                   |
|36660|$90.00 |You are GUARANTEED to love this                  |4.77                  |
|38605|$61.00 |SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST       |4.53                  |
|38610|$340.00|Short 

In [24]:
from pyspark.sql.functions import regexp_replace

price_num_df = listings \
    .withColumn('price_num', regexp_replace('price', '[$,]', '') .cast('float')) \

price_num_df.schema['price_num']

StructField('price_num', FloatType(), True)

In [25]:
price_num_df \
    .select('price_num', 'name') \
    .show(20, truncate = False)

+---------+-------------------------------------------------+
|price_num|name                                             |
+---------+-------------------------------------------------+
|70.0     |Holiday London DB Room Let-on going              |
|149.0    |Bright Chelsea  Apartment. Chelsea!              |
|411.0    |Very Central Modern 3-Bed/2 Bath By Oxford St W1 |
|NULL     |Battersea live/work artist house                 |
|210.0    |Bright 1 bedroom apt off brick lane in Shoreditch|
|280.0    |Kew Gardens 3BR house in cul-de-sac              |
|90.0     |You are GUARANTEED to love this                  |
|61.0     |SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST       |
|340.0    |Short Term Home                                  |
|49.0     |SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT         |
|NULL     |Stylish bedsit in Notting Hill ish flat.         |
|213.0    |2 Double bed apartment in quiet area North London|
|NULL     |Room in maisonette in chiswick                   |
|96.0   

In [26]:
price_num_df.filter((price_num_df.price_num < 100) & (price_num_df.review_scores_location < 4.5)) \
    .select('name', 'price', 'review_scores_location') \
    .show(truncate = False)

+--------------------------------------------------+------+----------------------+
|name                                              |price |review_scores_location|
+--------------------------------------------------+------+----------------------+
|NICE FAMILY HOME opposite NATURAL LANDSCAPED PARK |$40.00|4.36                  |
|A double Room 5mins from King's College Hospital  |$42.00|4.35                  |
|Penthouse Living in East London                   |$45.00|4.41                  |
|LOVELY & COZY ROOM WITH KITCHENETTE               |$23.00|4.31                  |
|Studio 20 min from center                         |$50.00|4.39                  |
|Spacious Double Room in East London               |$41.00|3.0                   |
|Stylish 2 Bed Apartment with Balcony              |$96.00|4.45                  |
|NICE DOUBLE ROOM IN A CLEAN  AND QUIET HOUSE      |$26.00|4.48                  |
|Vintage Oasis and Double Room and Peace           |$51.00|4.45                  |
|Sin

In [35]:
listings \
    .select(listings.property_type) \
    .distinct() \
    .show(truncate = False)

+----------------------------------+
|property_type                     |
+----------------------------------+
|Private room in lighthouse        |
|Private room in loft              |
|Private room in earthen home      |
|Entire chalet                     |
|Earthen home                      |
|Farm stay                         |
|Entire rental unit                |
|Shared room in hostel             |
|Shared room                       |
|Private room in condo             |
|Room in boutique hotel            |
|Private room in religious building|
|Room in bed and breakfast         |
|Private room in casa particular   |
|Private room in bungalow          |
|Entire cabin                      |
|Entire guesthouse                 |
|Hut                               |
|Private room in nature lodge      |
|Entire guest suite                |
+----------------------------------+
only showing top 20 rows


In [39]:
listings \
    .select(listings.property_type, listings.room_type) \
    .distinct() \
    .show(truncate = False)

+----------------------------------+---------------+
|property_type                     |room_type      |
+----------------------------------+---------------+
|Room in hostel                    |Hotel room     |
|Private room in casa particular   |Private room   |
|Dome                              |Entire home/apt|
|Entire serviced apartment         |Entire home/apt|
|Private room in loft              |Private room   |
|Private room in villa             |Private room   |
|Farm stay                         |Entire home/apt|
|Room in hotel                     |Hotel room     |
|Shared room in rental unit        |Shared room    |
|Private room in guest suite       |Private room   |
|Room in rental unit               |Hotel room     |
|Room in serviced apartment        |Hotel room     |
|Private room in serviced apartment|Private room   |
|Private room in hostel            |Private room   |
|Shared room                       |Shared room    |
|Private room in yurt              |Private ro

In [45]:
listings \
    .select(listings.property_type) \
    .distinct() \
    .write \
    .csv('data/property_types')